In [1]:
"""
PIPELINE STAGE: Demographic Data Parsing & Structural Normalization
PERIOD: 2020 - 2024
LIBRARIES: os, pandas, re

1. OBJECTIVE:
   Parse a non-standard, unstructured population CSV file, extract temporal and spatial data 
   points, and reshape it into a clean, relational DataFrame. This stage prepares the 
   demographic denominator necessary for subsequent per-capita energy consumption calculations.

2. FILE ARCHITECTURE:
   - Input (Raw Population): os.path.join("..", "..", "raw_data", "population", "population.csv")
   - Output (Cleaned Data): os.path.join("..", "..", "processed_data", "steps", "08_a_cleaned_population.xlsx")
   - Encoding: Utilizes 'iso-8859-9' to ensure Turkish character integrity during read operations.

3. CONTEXT-AWARE PARSING STRATEGY:
   A. Tokenization: Flattens the horizontal CSV by converting all newlines and carriage returns 
      into commas, creating a sequential 1D array of tokens.
   B. Anchor Detection (Regex): Scans tokens to identify anchor years using regex (^20\d{2}$). 
      Once a year is established, it cascades to all subsequent data blocks until a new year is found.
   C. Spatial Extraction: Identifies geographic clusters formatted as 'Province-Plate' (e.g., "ADANA-1"). 
      Splits these blocks to extract 'Province' and 'Plate', dynamically pairing them with the 
      immediate next numeric token representing the 'Population'.

4. DATA REFINEMENT & OUTPUT SPECIFICATIONS:
   - Schema Alignment: Enforces a strict column sequence: [Plate, Year, Province, Population].
   - Deduplication: Drops duplicate records based on the composite key [Plate, Year], keeping 
     the last valid entry to ensure database uniqueness.
   - Sorting Logic: Sorted strictly by 'Plate' (Ascending) and 'Year' (Ascending) to prime 
     the dataset for the final merge with the master energy database.
"""
import pandas as pd
import os
import re

def nufus_yapilandirma_v2():
    # Dosya Yolları
    IN_PATH = os.path.join("..", "..", "raw_data", "population", "population.csv")
    OUT_PATH = os.path.join("..", "..", "processed_data", "steps", "08_a_cleaned_population.xlsx")

   
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    records = []
    
    try:
        # Dosyayı oku ve içeriği normalize et
        with open(IN_PATH, 'r', encoding='iso-8859-9') as f:
            content = f.read()
            # Satır sonlarını virgüle çevirip temiz bir liste oluştur
            parts = [p.strip() for p in content.replace('\n', ',').replace('\r', ',').split(',') if p.strip()]

        current_year = None
        i = 0
        while i < len(parts):
            item = parts[i]
            
            # Yıl tespiti (Örn: 2020)
            if re.match(r'^20\d{2}$', item):
                current_year = int(item)
                i += 1
                continue
            
            # Veri bloğu tespiti (Örn: Adana-1)
            if current_year and '-' in item:
                if i + 1 < len(parts):
                    geo_info = item
                    pop_count = parts[i+1]
                    
                    try:
                        prov, plate = geo_info.split('-')
                        if pop_count.isdigit():
                            records.append({
                                'Plate': int(plate),      # 1. Sırada
                                'Year': current_year,     # 2. Sırada
                                'Province': prov.strip().upper(), # 3. Sırada
                                'Population': int(pop_count)     # 4. Sırada
                            })
                            i += 2
                            continue
                    except:
                        pass
            i += 1

        if not records:
            print("HATA: Veri işlenemedi.")
            return

        # DataFrame oluştur
        df_pop = pd.DataFrame(records)

        # 1. İSTEK: Sütun yerlerini değiştir (Plate, Year, Province, Population)
        # Liste yapısı ile sütun sırasını zorunlu kılıyoruz
        df_pop = df_pop[['Plate', 'Year', 'Province', 'Population']]
        
        # Mükerrerleri temizle
        df_pop = df_pop.drop_duplicates(subset=['Plate', 'Year'], keep='last')
        
        # 2. İSTEK: Plate ve Year sütunlarına göre artan (A-Z) sırala
        df_pop = df_pop.sort_values(by=['Plate', 'Year'], ascending=[True, True])
        
        # Excel'e aktar
        df_pop.to_excel(OUT_PATH, index=False)
        
        print("\n" + "="*40)
        print(f"İŞLEM BAŞARIYLA TAMAMLANDI")
        print(f"Toplam Satır: {len(df_pop)}")
        print(f"Sıralama: Plate ve Year bazlı artan")
        print(f"Dosya Yolu: {OUT_PATH}")
        print("="*40)

    except Exception as e:
        print(f"BEKLENMEDİK HATA: {e}")

if __name__ == "__main__":
    nufus_yapilandirma_v2()


İŞLEM BAŞARIYLA TAMAMLANDI
Toplam Satır: 405
Sıralama: Plate ve Year bazlı artan
Dosya Yolu: ..\..\processed_data\steps\08_a_cleaned_population.xlsx
